# Credit Card Approval Prediction using Machine Learning
### Academic Internship Project Jupyter Notebook

This notebook implements an end-to-end Machine Learning pipeline to predict whether a credit card application should be **Approved** or **Rejected** based on applicant profiles. It covers data loading, cleaning, exploratory data analysis, feature engineering, preprocessing, training four different algorithms, evaluation, model comparison, and model saving.

## Section 1: Data loading & Descriptive Statistics
We read the Excel dataset, check its shape, inspect data types, view sample rows, check missing values, and remove duplicates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the Excel file
df = pd.read_excel('credit_card_applicant_data.xlsx')
print(f"Dataset Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# First 10 rows
df.head(10)

In [ ]:
# Last 10 rows
df.tail(10)

In [ ]:
# Check missing values and duplicates
print("Missing Values:\n", df.isnull().sum())
print(f"\nDuplicate Records: {df.duplicated().sum()}")

# Drop duplicate records if any exist
df = df.drop_duplicates()
print(f"Shape after duplicate removal: {df.shape}")

In [ ]:
# Descriptive Statistics
df.describe(include='all')

In [ ]:
# Identify Numerical and Categorical Columns
numerical_cols = list(df.select_dtypes(include=['int64', 'float64']).columns)
categorical_cols = list(df.select_dtypes(include=['object']).columns)
target_col = 'Approval_Status'

print(f"Numerical columns: {numerical_cols}")
print(f"Categorical columns: {categorical_cols}")
print(f"Target Column: {target_col}")

## Section 2: Exploratory Data Analysis (EDA)
We visualize the distributions of the target column, income, credit history, and employment duration using SeaBorn and Matplotlib.

In [ ]:
sns.set_theme(style="whitegrid")

# 1. Approval Distribution
plt.figure(figsize=(6,4))
sns.countplot(x='Approval_Status', data=df, palette='Set2')
plt.title('Credit Card Approval Status Distribution')
plt.show()

# 2. Income Distribution
plt.figure(figsize=(8,5))
sns.histplot(x='Annual_Income', hue='Approval_Status', data=df, kde=True, multiple='stack')
plt.title('Annual Income Distribution by Approval Status')
plt.show()

In [ ]:
# 3. Credit History Distribution
plt.figure(figsize=(6,4))
sns.countplot(x='Credit_History', hue='Approval_Status', data=df, palette='viridis')
plt.title('Approval Status by Credit History')
plt.show()

# 4. Employment Duration Analysis
plt.figure(figsize=(7,5))
sns.boxplot(x='Approval_Status', y='Employment_Duration_Years', data=df, palette='pastel')
plt.title('Employment Duration vs Approval Status')
plt.show()

In [ ]:
# 5. Correlation Heatmap
plt.figure(figsize=(10,6))
num_df = df.select_dtypes(include=['int64', 'float64']).drop(columns=['Applicant_ID'], errors='ignore')
sns.heatmap(num_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.show()

## Section 3: Feature Engineering
We create helper columns: `Income_Group` (Low, Medium, High), `Employment_Duration_Category` (Short-term, Medium-term, Long-term), `Inquiries_to_Income_Ratio`, and numeric `Payment_History_Grade`.

In [ ]:
# Bin income
df['Income_Group'] = pd.cut(df['Annual_Income'], bins=[0, 50000, 110000, np.inf], labels=['Low', 'Medium', 'High'])

# Bin employment years
df['Employment_Duration_Category'] = pd.cut(df['Employment_Duration_Years'], bins=[-1, 2, 7, np.inf], labels=['Short-term', 'Medium-term', 'Long-term'])

# Inquiries-to-income ratio
df['Inquiries_to_Income_Ratio'] = df['Credit_Inquiries_Last_6M'] / (df['Annual_Income'] / 10000.0 + 1)

# Map payment history to numerical grade
pay_history_map = {'No Debt': 0, 'Late Payments': 1, 'Serious Default': 2}
df['Payment_History_Grade'] = df['Payment_History_Status'].map(pay_history_map)

print(df[['Income_Group', 'Employment_Duration_Category', 'Inquiries_to_Income_Ratio', 'Payment_History_Grade']].head())

## Section 4: Data Preprocessing
We separate the data into feature matrix `X` and target vector `y`, split them 80/20, and build a preprocessing pipeline using `ColumnTransformer` (scaling numericals and encoding categoricals).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Split features and target
X = df.drop(columns=['Applicant_ID', 'Approval_Status'])
y = df['Approval_Status'].map({'Approved': 1, 'Rejected': 0})

# Numeric and categorical features list
num_features = ['Age', 'Number_of_Children', 'Annual_Income', 'Employment_Duration_Years', 
                'Credit_Inquiries_Last_6M', 'Family_Size', 'Inquiries_to_Income_Ratio', 'Payment_History_Grade']
cat_features = ['Gender', 'Own_Car', 'Own_Property', 'Income_Type', 'Education_Level', 
                'Marital_Status', 'Housing_Type', 'Has_Work_Phone', 'Has_Phone', 'Has_Email', 
                'Credit_History', 'Income_Group', 'Employment_Duration_Category']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Transformers
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

print("Preprocessing pipeline created successfully.")

## Section 5 & 6: Model Building & Evaluation
We train Logistic Regression, Decision Tree, Random Forest, and XGBoost Classifiers, then evaluate them on the test set using standard validation metrics.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import time

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10),
    'XGBoost': XGBClassifier(random_state=42, n_estimators=100, max_depth=5, eval_metric='logloss')
}

comparison_results = []
trained_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    t0 = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - t0
    
    t0 = time.time()
    y_pred = pipe.predict(X_test)
    pred_time = time.time() - t0
    
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    trained_pipelines[name] = pipe
    comparison_results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'ROC-AUC': auc,
        'Train_Time': train_time,
        'Prediction_Time': pred_time
    })

comp_df = pd.DataFrame(comparison_results)
print(comp_df)

In [ ]:
# Show classification report for the best model
best_model_name = comp_df.loc[comp_df['F1'].idxmax(), 'Model']
best_pipe = trained_pipelines[best_model_name]
y_pred = best_pipe.predict(X_test)

print(f"\nClassification Report for Best Model: {best_model_name}\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## Section 7: Save Model
We save the best performing model pipeline as `model.pkl` using joblib so that it can be loaded for predictions in the Flask application.

In [ ]:
import joblib

# Save model pipeline
joblib.dump(best_pipe, 'model.pkl')
print(f"Saved {best_model_name} pipeline to 'model.pkl'")